In [1]:
import numpy as np
import nbimporter
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit
from cdt.data import load_dataset
from scipy.stats import gamma, norm
from sklearn.metrics import roc_auc_score
import os
import random
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from cdt.causality import pairwise
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import rankdata
from sklearn.linear_model import Lasso
import cepairsimplementation as ce
from scipy import stats
seedR = random.Random(42)
seedN = np.random.default_rng()

Detecting 1 CUDA device(s).


In [2]:
#RECI --- toolbox

#uses minmax scaling following by reashaping between -1 and 1
#uses a 3rd degree polynomial

def reci_score(c,e,degree,cdt):
    scaler = MinMaxScaler(feature_range=(0,1))
    c = scaler.fit_transform(c)
    e = scaler.fit_transform(e)
    poly = PolynomialFeatures(degree=degree)
    poly_c = poly.fit_transform(c)
    regressor = LinearRegression()
    regressor.fit(poly_c, e)
    if cdt:
        # Get the exponents for each term; each row corresponds to a column in poly_c
        powers = poly.powers_
        # Compute the total degree for each term by summing across the features
        degrees = powers.sum(axis=1)
        # Zero out the linear (degree=1) and quadratic (degree=2) terms
        for i, d in enumerate(degrees):
            if d == 1 or d == 2:
                poly_c[:, i] = 0
    
    y_predict = regressor.predict(poly_c)
    error = mean_squared_error(y_predict, e)
    return -error

def reci_score_sparse(c,e,degree,cdt):
    scaler = MinMaxScaler(feature_range=(0,1))
    c = scaler.fit_transform(c)
    e = scaler.fit_transform(e)
    poly = PolynomialFeatures(degree=degree)
    poly_c = poly.fit_transform(c)
    regressor = Lasso()
    regressor.fit(poly_c, e)
    y_predict = regressor.predict(poly_c)
    error=np.sum(np.abs(y_predict - e))
    return -error

def RECI(d, degree=1,cdt=False):
    x,y=d
    return reci_score(x,y,degree,cdt)-reci_score(y,x, degree, cdt)

def remove_outliers(x, y,per=0.9):
    x, y = np.array(x), np.array(y)

    # Compute IQR for x and y
    Q1_x, Q3_x = np.percentile(x, [(1-per)/2, per+(1-per)/2])
    #IQR_x = Q3_x - Q1_x
    #lower_x, upper_x = Q1_x - mul * IQR_x, Q3_x + mul* IQR_x
    lower_x,upper_x=Q1_x,Q3_x

    Q1_y, Q3_y = np.percentile(y, [(1-per)/2, per+(1-per)/2])
    #IQR_y = Q3_y - Q1_y
    #lower_y, upper_y = Q1_y - mul * IQR_y, Q3_y + mul * IQR_y
    lower_y,upper_y=Q1_y,Q3_y

    # Create a mask for valid (non-outlier) points
    mask = (x >= lower_x) & (x <= upper_x) & (y >= lower_y) & (y <= upper_y)
    print(len(x[mask]))
    return x[mask], y[mask]

def RECIinliers(d, degree=1,cdt=False):
    x,y=d
    if x.shape[1]>1 or y.shape[1]>1:
        return np.nan
    x,y=remove_outliers(x,y)
    x=x.reshape(-1,1)
    y=y.reshape(-1,1)
    return reci_score(x,y,degree,cdt)-reci_score(y,x, degree, cdt)



In [3]:
ce.test_tuebingen(RECI)

(np.float64(0.7226242354145286), np.float64(0.7019660032976133))